## Data Loading and Preparation

In [ ]:
import pandas as pd
import numpy as np
import itertools
import warnings
from typing import Any
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from tiportfolio.metrics import compute_metrics
from tiportfolio.allocation import DollarNeutral
from tiportfolio.data import fetch_prices
from tiportfolio.backtest import run_backtest
from tiportfolio.engine import ScheduleBasedEngine
from tiportfolio.calendar import Schedule

In [ ]:
# Prepare Universe stocks for dynamic stock selection
stock_universe = ['AAPL', 'MSFT', 'NVDA', 'INTC', 'AMD', 'KO', 'PEP', 'PG', 'JNJ', 'WMT', 'META', 'AMZN', 'GOOGL', 'TSLA', 'NFLX']

CASH   = "BIL"   
START  = '2018-01-01' 
END    = '2024-12-31'
INITIAL_VALUE = 100000
LOOKBACK = 21
TOP_N = 2    
BOOK_SIZE = 0.5
TOLERANCE = 0.05
FEE = 0.0035

all_symbols = stock_universe + [CASH]

## Define dynamic stock selection long/short strategy by Price-based Factors
##### 1. Momentum

In [104]:
class DynamicMomentumDollarNeutral(DollarNeutral):
    """
    On each rebalancing day, review the returns over the past N days.
    Go long on the top_n best-performing stocks and short on the bottom_n worst-performing stocks.
    Maintain a neutral cash position.
    """
    def __init__(self, universe: list[str], cash_symbol: str, 
                 lookback_window: int = 21, top_n: int = 2, 
                 book_size: float = 0.5, tolerance: float = 0.05):
        
        self.universe = universe
        self.lookback_window = lookback_window
        self.top_n = top_n
        
        super().__init__(
            long_weights={universe[0]: 1.0},
            short_weights={universe[1]: 1.0},
            cash_symbol=cash_symbol,
            book_size=book_size,
            tolerance=tolerance
        )

    def get_symbols(self) -> list[str]:
        """
        Tell Engine which stock price data our strategy needs.
        Send back the entire Univers because dynamically selecting stocks.
        """
        return self.universe + [self.cash_symbol]

    def get_target_weights(self, date: pd.Timestamp, total_equity: float, 
                           positions_dollars: dict[str, float], 
                           prices_row: pd.Series, **context: Any) -> dict[str, float]:
        
        prices_history = context.get("prices_history")
        
        # 1. Check if the historical data is sufficient
        if prices_history is None or len(prices_history) < self.lookback_window:
            # Empty-handed when there is insufficient information
            self.long_weights = {self.universe[0]: 0.0}
            self.short_weights = {self.universe[1]: 0.0}
            self.long_book_size = 0.0
            self.short_book_size = 0.0
            return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

        # 2. Calculate the cumulative rate of return over the past lookback_window days
        # Extract the historical prices of Universe stocks
        hist_prices = prices_history[self.universe].tail(self.lookback_window)
        # Rate of return = (Today's price / Price N days ago) - 1
        returns = (hist_prices.iloc[-1].astype(float) / hist_prices.iloc[0].astype(float)) - 1
        
        # 3. Sort to identify winners and losers
        returns = returns.dropna()
        
        if len(returns) >= self.top_n * 2:
            sorted_returns = returns.sort_values(ascending=False)
            
            # Find the N stocks with the largest price changes
            winners = sorted_returns.head(self.top_n).index.tolist()
            losers = sorted_returns.tail(self.top_n).index.tolist()
            
            # 4. Distribute weights evenly
            # If 2 are selected, each will account for 0.5 (50%) of the Long Book.
            weight_per_stock = 1.0 / self.top_n
            
            self.long_weights = {sym: weight_per_stock for sym in winners}
            self.short_weights = {sym: weight_per_stock for sym in losers}
            
            # Restore normal fund size
            self.long_book_size = self.book_size
            self.short_book_size = self.book_size
            
            print(f"[{date.date()}] Long: {winners} | Short: {losers}")
            
        else:
            # Insufficient stock quantity, remain uninvested.
            self.long_book_size = 0.0
            self.short_book_size = 0.0

        # 5. Calling the parent category to handle tolerance and cash allocation
        return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

##### 2. Volatility

In [105]:
class DynamicVolatilityDollarNeutral(DollarNeutral):
    """
    Low-Volatility Anomaly Strategy: 
    Long the N stocks with the lowest realized volatility.
    Short the N stocks with the highest realized volatility.
    Maintains a dollar-neutral exposure.
    """
    def __init__(self, universe: list[str], cash_symbol: str, 
                 lookback_window: int = 63, top_n: int = 2, 
                 book_size: float = 0.5, tolerance: float = 0.05):
        
        self.universe = universe
        self.lookback_window = lookback_window # Usually 3 months (63 trading days)
        self.top_n = top_n
        
        # Initialize parent class with dummy weights
        super().__init__(
            long_weights={universe[0]: 1.0},
            short_weights={universe[1]: 1.0},
            cash_symbol=cash_symbol,
            book_size=book_size,
            tolerance=tolerance
        )

    def get_symbols(self) -> list[str]:
        # Return entire universe to ensure engine fetches all necessary data
        return self.universe + [self.cash_symbol]

    def get_target_weights(self, date: pd.Timestamp, total_equity: float, 
                           positions_dollars: dict[str, float], 
                           prices_row: pd.Series, **context: Any) -> dict[str, float]:
        
        prices_history = context.get("prices_history")
        
        # Check for sufficient historical data
        if prices_history is None or len(prices_history) < self.lookback_window + 1:
            self.long_book_size = 0.0
            self.short_book_size = 0.0
            return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

        # 1. Calculate annualized realized volatility for each stock in the universe
        hist_prices = prices_history[self.universe].tail(self.lookback_window + 1)
        daily_returns = hist_prices.pct_change().dropna()
        
        # Annualized Volatility = standard deviation * sqrt(252)
        vols = daily_returns.std() * np.sqrt(252)
        vols = vols.dropna().sort_values(ascending=False)

        # 2. Rank and select Winners (Low Vol) and Losers (High Vol)
        if len(vols) >= self.top_n * 2:
            # High Volatility stocks (to be Shorted)
            high_vol_stocks = vols.head(self.top_n).index.tolist()
            # Low Volatility stocks (to be Longed)
            low_vol_stocks = vols.tail(self.top_n).index.tolist()
            
            # 3. Assign equal weights within each book
            weight_per_stock = 1.0 / self.top_n
            
            # Strategy: Long Low Vol, Short High Vol
            self.long_weights = {sym: weight_per_stock for sym in low_vol_stocks}
            self.short_weights = {sym: weight_per_stock for sym in high_vol_stocks}
            
            self.long_book_size = self.book_size
            self.short_book_size = self.book_size
            
            # Optional: print for debugging
            # print(f"[{date.date()}] Long (Low Vol): {low_vol_stocks} | Short (High Vol): {high_vol_stocks}")
        else:
            self.long_book_size = 0.0
            self.short_book_size = 0.0

        # 4. Delegate to parent class for final weight calculation and cash management
        return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

##### 3. Cointegration (Best Statistical Pair over full period)

In [106]:
class CointegrationDollarNeutral(DollarNeutral):
    """
    A money-neutral pair trading strategy based on co-integration and Z-Score signals.
    """
    def __init__(self, symbol_A: str, symbol_B: str, cash_symbol: str, 
                 hedge_ratio: float, lookback_window: int = 252, 
                 z_entry: float = 2.0, z_exit: float = 0.0, 
                 book_size: float = 0.5, tolerance: float = 0.05):
        
        self.symbol_A = symbol_A
        self.symbol_B = symbol_B
        self.hedge_ratio = hedge_ratio
        self.lookback_window = lookback_window
        self.z_entry = z_entry
        self.z_exit = z_exit
        self.target_book_size = book_size # The proportion of funds to be allocated to the storage target

        # 0: neutral, 1: long spread, -1: short spread
        self.current_signal = 0 
        
        super().__init__(
            long_weights={symbol_A: 1.0},
            short_weights={symbol_B: 1.0},
            cash_symbol=cash_symbol,
            book_size=0.0, 
            tolerance=tolerance
        )

    def get_symbols(self) -> list[str]:
        # Fetch prices for the two symbols and cash symbol
        return [self.symbol_A, self.symbol_B, self.cash_symbol]

    def get_target_weights(self, date: pd.Timestamp, total_equity: float, 
                           positions_dollars: dict[str, float], 
                           prices_row: pd.Series, **context: Any) -> dict[str, float]:
        """
        Calculate Z-Score -> 
        Determine signal -> 
        Dynamically adjust the long/short target and capital ratio -> 
        Assign final weight to parent category.
        """
        # Retrieve historical prices from the context
        prices_history = context.get("prices_history")
        
        # Maintain cash position if history is insufficient
        if prices_history is None or len(prices_history) < self.lookback_window:
            self.long_book_size = 0.0
            self.short_book_size = 0.0
            return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

        # Extract historical prices for the specified window
        hist = prices_history[[self.symbol_A, self.symbol_B]].tail(self.lookback_window)
        log_hist_A = np.log(hist[self.symbol_A])
        log_hist_B = np.log(hist[self.symbol_B])

        # Calculate historical spread using hedge ratio
        historical_spreads = log_hist_A - (self.hedge_ratio * log_hist_B)
        
        spread_mean = historical_spreads.mean()
        spread_std = historical_spreads.std()
        
        # Calculate current Z-Score
        if spread_std > 0:
            current_log_A = np.log(prices_row[self.symbol_A])
            current_log_B = np.log(prices_row[self.symbol_B])
            current_spread = current_log_A - (self.hedge_ratio * current_log_B)
            z_score = (current_spread - spread_mean) / spread_std
        else:
            z_score = 0.0

         # Determine trading signal based on Z-Score thresholds
        if z_score > self.z_entry:
            self.current_signal = -1  # Excessive price spread: Short A, Long B
        elif z_score < -self.z_entry:
            self.current_signal = 1   # Price spread too small: Long A, Short B
        elif abs(z_score) < self.z_exit:
            self.current_signal = 0   # Regression to the mean: Close position and go empty

        # Assign book sizes and weights based on signal
        if self.current_signal == 1:
            self.long_weights = {self.symbol_A: 1.0}
            self.short_weights = {self.symbol_B: 1.0}
            self.long_book_size = self.target_book_size
            self.short_book_size = self.target_book_size            
        elif self.current_signal == -1:
            self.long_weights = {self.symbol_B: 1.0}
            self.short_weights = {self.symbol_A: 1.0}
            self.long_book_size = self.target_book_size
            self.short_book_size = self.target_book_size            
        else:
            # The parent category will automatically allocate all funds to cash_symbol
            self.long_book_size = 0.0
            self.short_book_size = 0.0

        return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

In [107]:
def test_johansen_cointegration(log_prices_data, det_order=0, k_ar_diff=1):
    """
    Perform Johansen cointegration test on log price data.
    
    Parameters:
    -----------
    log_prices_data : pd.DataFrame
        DataFrame with log prices for multiple assets
    det_order : int, default 0
        Deterministic order (0: no constant, 1: constant, 2: constant + trend)
    k_ar_diff : int, default 1
        Number of lagged differences in the model
    
    Returns:
    --------
    dict
        Dictionary containing test results:
        - 'eigenvectors': Eigenvectors from Johansen test
        - 'eigenvalues': Eigenvalues from Johansen test
        - 'trace_stat': Trace statistics
        - 'max_eig_stat': Maximum eigenvalue statistics
        - 'trace_crit': Trace critical values
        - 'max_eig_crit': Max eigenvalue critical values
        - 'coint_rank': Number of cointegrating relationships
    """
    try:
        # Perform Johansen cointegration test
        result = coint_johansen(log_prices_data, det_order, k_ar_diff)
        
        # Extract results
        eigenvectors = result.evec
        eigenvalues = result.eig
        trace_stat = result.lr1
        max_eig_stat = result.lr2
        trace_crit = result.cvt
        max_eig_crit = result.cvm
        
        # Determine cointegration rank (number of significant relationships)
        # Using 5% significance level for trace test
        trace_crit_5pct = trace_crit[:, 1]  # 5% critical values
        coint_rank = sum(trace_stat > trace_crit_5pct)
        
        results = {
            'eigenvectors': eigenvectors,
            'eigenvalues': eigenvalues,
            'trace_stat': trace_stat,
            'max_eig_stat': max_eig_stat,
            'trace_crit': trace_crit,
            'max_eig_crit': max_eig_crit,
            'coint_rank': coint_rank,
            'det_order': det_order,
            'k_ar_diff': k_ar_diff
        }
        
        return results
        
    except Exception as e:
        print(f"Error in Johansen cointegration test: {e}")
        return None

## Find the best pairs trading by Price-based Factors
 - Momentum (Highest & Lowest Returns)
 - Volatility (Highest & Lowest Risk)
 - Cointegration (Best Statistical Pair over full period)


In [ ]:
def run_comprehensive_scanner(prices_df: pd.DataFrame, lookback_days: int = 252) -> dict:
    """
    Scans the given universe of stocks across four dimensions:
    1. Momentum (Highest Return)
    2. High Volatility
    3. Low Volatility
    4. Cointegration Pairs
    
    Args:
        prices_df: DataFrame with date index and symbols as columns (Close prices)
        lookback_days: Number of recent days to use for Momentum and Volatility calculations
    
    Returns:
        dict: A dictionary containing the best candidates for each category.
    """
    print(f"Running Comprehensive Market Scanner (Lookback: {lookback_days} days)...")
    results = {}
    
    # Use the most recent 'lookback_days' for Momentum and Volatility
    recent_prices = prices_df.tail(lookback_days)
    
    # Dimension 1: Momentum (Returns)
    # Return = (Last Price / First Price) - 1
    returns = (recent_prices.iloc[-1] / recent_prices.iloc[0]) - 1
    returns = returns.dropna().sort_values(ascending=False)
    
    results['momentum'] = {
        'top_winners': returns.head(3).to_dict(),
        'top_losers': returns.tail(3).to_dict()
    }
    
    # Dimension 2 : Volatility
    # Daily returns
    daily_returns = recent_prices.pct_change().dropna()
    # Annualized Volatility = std * sqrt(252)
    annualized_vol = daily_returns.std() * np.sqrt(252)
    annualized_vol = annualized_vol.dropna().sort_values(ascending=False)
    
    results['volatility'] = {
        'highest_vol': annualized_vol.head(3).to_dict(),
        'lowest_vol': annualized_vol.tail(3).to_dict()
    }
    
    # Dimension 3: Cointegration
    # Cointegration usually requires a longer history to be stable, 
    # so we use the full prices_df instead of just recent_prices.
    valid_symbols = prices_df.columns.tolist()
    pairs = list(itertools.combinations(valid_symbols, 2))
    
    best_pair = None
    best_score = 0.0
    best_hedge_ratio = 1.0
    
    warnings.filterwarnings("ignore")
    
    for sym_a, sym_b in pairs:
        pair_data = prices_df[[sym_a, sym_b]].dropna()
        if len(pair_data) < 252:
            continue
            
        log_prices = np.log(pair_data)
        try:
            coint_res = coint_johansen(log_prices, det_order=0, k_ar_diff=1)
            score = coint_res.lr1[0] / coint_res.cvt[0, 1] 
            
            if score > 1.0 and score > best_score:
                best_score = score
                best_pair = (sym_a, sym_b)
                eigenvector = coint_res.evec[:, 0]
                best_hedge_ratio = - (eigenvector[1] / eigenvector[0])
        except Exception:
            continue
            
    warnings.filterwarnings("default")
    
    results['cointegration'] = {
        'best_pair': best_pair,
        'score': best_score,
        'hedge_ratio': best_hedge_ratio
    }
    
    return results

# ==========================================
# Execution Workflow
# ==========================================
print("Fetching historical data for scanner...")
try:
    # 1. Fetch ALL data (including BIL for the engine)
    all_data = fetch_prices(all_symbols, start=START, end=END)
    
    # 2. Prepare close_prices ONLY for the stocks (excluding BIL)
    stock_close_prices = pd.DataFrame({
        sym: df['close'] for sym, df in all_data.items() if sym in stock_universe
    }).dropna()

    print("Data fetch complete.")
except Exception as e:
    print(f"Data fetch failed: {e}")
    raise

# Run the scanner
scan_results = run_comprehensive_scanner(stock_close_prices, lookback_days=252)

print("="*60)

print("1. MOMENTUM (Highest & Lowest Returns)")
print("Top Winners:")
for sym, ret in scan_results['momentum']['top_winners'].items():
    print(f"   - {sym}: {ret*100:+.2f}%")
print("Top Losers:")
for sym, ret in scan_results['momentum']['top_losers'].items():
    print(f"   - {sym}: {ret*100:+.2f}%")

print("\n2. VOLATILITY (Highest & Lowest Risk)")
print("Highest Volatility:")
for sym, vol in scan_results['volatility']['highest_vol'].items():
    print(f"   - {sym}: {vol*100:.2f}% (Annualized)")
print("Lowest Volatility:")
for sym, vol in scan_results['volatility']['lowest_vol'].items():
    print(f"   - {sym}: {vol*100:.2f}% (Annualized)")

print("\n3. COINTEGRATION (Best Statistical Pair over full period)")
pair = scan_results['cointegration']['best_pair']
score = scan_results['cointegration']['score']
hr = scan_results['cointegration']['hedge_ratio']

if pair:
    print(f"   Best Pair: {pair[0]} & {pair[1]}")
    print(f"   Score:     {score:.4f} (>1.0 is significant)")
    print(f"   Hedge Ratio: {hr:.4f}")
else:
    print("   No significant cointegrated pairs found.")

Fetching historical data for scanner...
Loading bar data...
Loaded bar data: 0:00:03 

Data fetch complete.
Running Comprehensive Market Scanner (Lookback: 252 days)...
1. MOMENTUM (Highest & Lowest Returns)
Top Winners:
   - NVDA: +177.73%
   - NFLX: +84.93%
   - WMT: +74.40%
Top Losers:
   - PEP: -7.79%
   - AMD: -16.94%
   - INTC: -60.02%

2. VOLATILITY (Highest & Lowest Risk)
Highest Volatility:
   - TSLA: 63.52% (Annualized)
   - NVDA: 52.53% (Annualized)
   - INTC: 51.26% (Annualized)
Lowest Volatility:
   - JNJ: 15.12% (Annualized)
   - PG: 15.02% (Annualized)
   - KO: 12.82% (Annualized)

3. COINTEGRATION (Best Statistical Pair over full period)
   Best Pair: AAPL & JNJ
   Score:     1.4124 (>1.0 is significant)
   Hedge Ratio: 4.4880


## Backtesting strategy comparison of Price-based Factors
##### 1. Momentum (Highest & Lowest Returns)

In [121]:
# Perform backtesting (monthly portfolio rebalancing)
strategy_mom = DynamicMomentumDollarNeutral(
    universe=stock_universe,
    cash_symbol=CASH,
    lookback_window=21,  # returns over the past 21 days
    top_n=2,             # top 2 long & bottom 2 short positions
    book_size=0.5,
    tolerance=0.05
)

engine_mom = ScheduleBasedEngine(
    allocation=strategy_mom,
    rebalance=Schedule('month_end'),
    initial_value=INITIAL_VALUE,
    fee_per_share=FEE
)

try:
    result_mom = engine_mom.run(
        symbols=all_symbols,
        start=START,
        end=END,
        prices_df=all_data 
    )
    
    metrics_mom = compute_metrics(result_mom.equity_curve)
    
    print("\n" + "="*60)
    print("🏆 DYNAMIC MOMENTUM STRATEGY RESULTS")
    print("="*60)
    
    print(f"Sharpe Ratio:   {metrics_mom.get('sharpe_ratio', 0):.3f}")
    print(f"Sortino Ratio:  {metrics_mom.get('sortino_ratio', 0):.3f}")
    print(f"MAR Ratio:      {metrics_mom.get('mar_ratio', 0):.3f}")
    print(f"CAGR:           {metrics_mom.get('cagr', 0):.2%}")
    print(f"Max Drawdown:   {metrics_mom.get('max_drawdown', 0):.2%}")
    print(f"Kelly Leverage: {metrics_mom.get('kelly_leverage', 0):.3f}")
    print(f"Mean Excess Return: {metrics_mom.get('mean_excess_return', 0):.3f}")
    print(f"Annual Return:  {metrics_mom.get('annualized_return', 0):.2%}")
    
    trade_count = 0
    prev_weights = None
    for dec in result_mom.rebalance_decisions:
        curr_weights = getattr(dec, 'target_weights', {})
        if prev_weights is not None and curr_weights != prev_weights:
            trade_count += 1
        prev_weights = curr_weights
        
    print(f"Number of Trade Events: {trade_count}")
    print("="*60)

except Exception as e:
    print(f"❌ Backtest failed: {e}")

[2018-02-28] Long: ['AMZN', 'AAPL'] | Short: ['KO', 'WMT']
[2018-03-29] Long: ['KO', 'INTC'] | Short: ['AMD', 'TSLA']
[2018-04-30] Long: ['AMD', 'TSLA'] | Short: ['PEP', 'PG']
[2018-05-31] Long: ['AMD', 'NFLX'] | Short: ['WMT', 'JNJ']
[2018-06-29] Long: ['TSLA', 'NFLX'] | Short: ['NVDA', 'INTC']
[2018-07-31] Long: ['AMD', 'GOOGL'] | Short: ['NFLX', 'TSLA']
[2018-08-31] Long: ['AMD', 'NVDA'] | Short: ['KO', 'TSLA']
[2018-09-28] Long: ['AMD', 'NFLX'] | Short: ['GOOGL', 'INTC']
[2018-10-31] Long: ['TSLA', 'WMT'] | Short: ['NVDA', 'AMD']
[2018-11-30] Long: ['AMD', 'PEP'] | Short: ['AAPL', 'NVDA']
[2018-12-31] Long: ['PG', 'META'] | Short: ['NVDA', 'AMD']
[2019-01-31] Long: ['NFLX', 'AMD'] | Short: ['PEP', 'TSLA']
[2019-02-28] Long: ['AMD', 'NVDA'] | Short: ['WMT', 'KO']
[2019-03-29] Long: ['NVDA', 'AAPL'] | Short: ['WMT', 'TSLA']
[2019-04-30] Long: ['META', 'GOOGL'] | Short: ['INTC', 'TSLA']
[2019-05-31] Long: ['AMD', 'KO'] | Short: ['TSLA', 'NVDA']
[2019-06-28] Long: ['TSLA', 'NVDA'] | Sh

##### 2. Volatility (Highest & Lowest Risk)


##### 3. Cointegration (Best Statistical Pair over full period)